# Fase 1 - EDA para revisión de datos

## Objetivo
Explorar y revisar los datos del proyecto de accesibilidad joven (UK) para validar distribuciones, detectar outliers y entender las relaciones entre variables antes de cualquier modelado.

## Insumos
- `data/integrado/maestro_joven_uk.parquet`
- `data/limpia/hipotecas_limpio.parquet`

## Cálculo del esfuerzo
Esfuerzo = cuota hipotecaria mensual / salario joven mensual

Se usa `indicadores.cuota_mensual_pmt()` con:
- Tasa media hipotecas limpia: 6.043%
- Comisiones media: 1115.78 GBP
- LTV: 0.90 (90%)
- Plazo: 25 años

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from proyecto import config
from proyecto import indicadores
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Cargar datos
maestro = pd.read_parquet(config.RUTA_MAESTRO).copy()
hipotecas = pd.read_parquet(config.RUTA_HIPOTECAS_LIMPIO).copy()

print("Dimensiones maestro:", maestro.shape)
print("Regiones:", maestro['region'].nunique())
print("Años:", maestro['anio'].min(), "-", maestro['anio'].max())
print("Clave (anio, region) única:", maestro.duplicated(['anio','region']).sum() == 0)

# Calcular esfuerzo
tasa = float(hipotecas['tasa_inicial_pct'].mean())
comisiones = float(hipotecas['comisiones_total'].mean())
print(f"\nTasa media: {tasa:.3f}% | Comisiones: {comisiones:.2f}")

df = maestro.copy()
df['cuota_mensual'] = df.apply(
    lambda r: indicadores.cuota_mensual_pmt(r['precio_mediana'], tasa, config.PLAZO_ANIOS, config.LTV, comisiones),
    axis=1
)
df['salario_joven_mensual'] = df['salario_joven_region'] / 12.0
df['esfuerzo'] = df['cuota_mensual'] / df['salario_joven_mensual']

print(f"\nEsfuerzo: min={df['esfuerzo'].min():.3f} | mediana={df['esfuerzo'].median():.3f} | max={df['esfuerzo'].max():.3f}")
print(f"Filas > 0.30: {int((df['esfuerzo'] > 0.30).sum())} / {len(df)}")

## 1.1 Histograma del esfuerzo
Vemos la distribución del esfuerzo hipotecario por todas las observaciones.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.6))
datos = df['esfuerzo'].dropna()
ax.hist(datos, bins=20, color='#0e7490', edgecolor='white')
ax.axvline(datos.mean(), color='#dc2626', ls='--', lw=2, label=f"Media {datos.mean():.3f}")
ax.axvline(datos.median(), color='#059669', ls='-', lw=2, label=f"Mediana {datos.median():.3f}")
ax.axvline(0.30, color='#7c3aed', ls=':', lw=2, label="Umbral 0.30")
ax.set_xlabel("Esfuerzo (cuota/salario joven mensual)")
ax.set_ylabel("Frecuencia")
ax.set_title("Distribución del esfuerzo hipotecario")
ax.legend()
plt.tight_layout()
plt.savefig(config.MD_FIGURAS / "rev_histograma_esfuerzo.png", dpi=110)
plt.show()

## 1.2 Boxplot del esfuerzo
Detecta visualmente valores atípicos.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2.8))
ax.boxplot(df['esfuerzo'].dropna(), vert=False, widths=0.55)
ax.axvline(0.30, color='#7c3aed', ls=':', lw=2, label="Umbral 0.30")
ax.set_xlabel("Esfuerzo (cuota/salario joven mensual)")
ax.set_title("Boxplot del esfuerzo")
ax.legend()
plt.tight_layout()
plt.savefig(config.MD_FIGURAS / "rev_boxplot_esfuerzo.png", dpi=110)
plt.show()

## 1.3 Regla IQR para esfuerzo y precio
Calcula Q1, Q3, IQR y límites para detectar atípicos según la regla de Tukey (1.5 * IQR).

In [ ]:
def regla_iqr(s):
    s_clean = s.dropna()
    if len(s_clean) == 0:
        return {}
    q1 = s_clean.quantile(0.25)
    q3 = s_clean.quantile(0.75)
    iqr = q3 - q1
    lim_sup = q3 + 1.5 * iqr
    lim_inf = q1 - 1.5 * iqr
    return {
        "q1": float(q1),
        "q3": float(q3),
        "iqr": float(iqr),
        "lim_inf": float(lim_inf),
        "lim_sup": float(lim_sup),
        "out_sup": int((s_clean > lim_sup).sum()),
        "out_inf": int((s_clean < lim_inf).sum()),
    }

for col in ["esfuerzo", "precio_mediana"]:
    iq = regla_iqr(df[col])
    print(f"{col}: Q1={iq['q1']:.4f} Q3={iq['q3']:.4f} IQR={iq['iqr']:.4f}")
    print(f"  límites: [{iq['lim_inf']:.4f}, {iq['lim_sup']:.4f}] | outliers sup={iq['out_sup']} inf={iq['out_inf']} tot={iq['outliers_tot']}")

## 1.4 Esfuerzo por región
Compara el esfuerzo medio y la mediana de años de salario entre regiones.

In [ ]:
agg_reg = df.groupby("region", observed=True).agg(
    esfuerzo_medio=("esfuerzo", "mean"),
    años_salario_medio=("anos_salario", "mean"),
    precio_mediana_medio=("precio_mediana", "mean"),
    n=("esfuerzo", "count"),
).reset_index().sort_values("esfuerzo_medio", ascending=False)

print(agg_reg.round(3).to_string(index=False))

fig, ejes = plt.subplots(1, 2, figsize=(12, 4.8))
agg_reg.plot.barh(x="region", y="esfuerzo_medio", ax=ejes[0], color="#dc2626")
ejes[0].axvline(0.30, color="#7c3aed", ls=":", lw=2)
ejes[0].set_title("Esfuerzo medio por región")
ejes[0].set_xlabel("Esfuerzo")

agg_reg.plot.barh(x="region", y="años_salario_medio", ax=ejes[1], color="#059669")
ejes[1].set_title("Años de salario medio por región")
ejes[1].set_xlabel("Años de salario")
plt.tight_layout()
plt.savefig(config.MD_FIGURAS / "rev_barras_esfuerzo_anos_salario_region.png", dpi=110)
plt.show()

## 1.5 Mapa de calor de correlaciones
Correlación de Pearson entre las variables numéricas clave y el esfuerzo.

In [ ]:
cols_corr = ["precio_mediana", "salario_joven_region", "anos_salario", "precio_var_pct",
             "tmax_anom", "tmin_anom", "lluvia_anom", "sol_anom", "af_anom", "esfuerzo"]
corr = df[cols_corr].corr()
plt.figure(figsize=(9, 7))
im = plt.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
plt.colorbar(im, label="Correlación Pearson")
plt.xticks(range(len(cols_corr)), cols_corr, rotation=45, ha="right")
plt.yticks(range(len(cols_corr)), cols_corr)
for i in range(len(cols_corr)):
    for j in range(len(cols_corr)):
        plt.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center", fontsize=7)
plt.title("Mapa de calor de correlaciones")
plt.tight_layout()
plt.savefig(config.MD_FIGURAS / "rev_heatmap_correlaciones.png", dpi=110)
plt.show()

## 1.6 Matriz de dispersión
Relaciones par a par entre las principales variables numéricas.

In [ ]:
from pandas.plotting import scatter_matrix

cols_sm = ["precio_mediana", "salario_joven_region", "anos_salario", "esfuerzo"]
scatter_matrix(df[cols_sm].dropna(), figsize=(9, 9), diagonal="hist", alpha=0.5, color="#0e7490")
plt.suptitle("Matriz de dispersión", y=0.92)
plt.tight_layout()
plt.savefig(config.MD_FIGURAS / "rev_matriz_dispersion.png", dpi=110)
plt.show()

## Conclusión Fase 1
- 60 filas (10 regiones × 6 años 2015-2020), clave única.
- Esfuerzo entre 0.42 y 1.07; **todas las filas superan el umbral 0.30**.
- No hay atípicos por IQR en el esfuerzo. Londres y Sureste son las regiones con mayor esfuerzo.